# 3. Model Training: Tiny Transformer Multimodal Fusion
This notebook trains our Method B student: a tiny multi-layer Transformer Encoder that treats visual and audio representations as sequence tokens, prepends a `[CLS]` token, and projects the final state to the teacher embedding space.


In [1]:
# Robust path and import setup
import os
import sys
for p in ['.', '..', '../..', '/kaggle/working']:
    abs_path = os.path.abspath(p)
    if abs_path not in sys.path:
        sys.path.append(abs_path)

def find_file(filename):
    # Check standard paths
    possible_paths = [
        filename,
        os.path.join("..", filename),
        os.path.join("features", filename),
        os.path.join("..", "features", filename),
        os.path.join("features", "patched_features", filename),
        os.path.join("..", "features", "patched_features", filename),
        os.path.join("models", filename),
        os.path.join("..", "models", filename),
        os.path.join("/kaggle/working", filename)
    ]
    for p_path in possible_paths:
        if os.path.exists(p_path):
            return p_path
    # Check /kaggle/input recursively for attached datasets
    if os.path.exists("/kaggle/input"):
        import glob
        matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if matches:
            return matches[0]
    return filename

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Add paths to sys.path to find 'src' modular code in any execution directory

# Helper to find files in different directories (local vs Kaggle setup)
from src.models import TransformerFusionApproximator
from src.loss import InfoNCELoss
from src.dataset import MultimodalEmbeddingDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cpu


## Step 1: Load datasets


In [2]:
train_dataset = MultimodalEmbeddingDataset(file_path=find_file("train_features.pt"))
test_dataset = MultimodalEmbeddingDataset(file_path=find_file("test_features.pt"))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


## Step 2: Train Method B (Tiny Transformer Fusion)
We optimize the model using a joint loss: $L = L_{cos} + 0.5 \cdot L_{InfoNCE}$ to combine absolute alignment and relative cross-modal contrast.


In [3]:
model_trans = TransformerFusionApproximator(img_dim=512, aud_dim=128, embed_dim=256, num_heads=4, num_layers=2, output_dim=1024).to(device)
optimizer = optim.AdamW(model_trans.parameters(), lr=1e-3, weight_decay=1e-4)

cosine_loss_fn = lambda pred, target: (1 - torch.nn.functional.cosine_similarity(pred, target)).mean()
infonce_loss_fn = InfoNCELoss(temperature=0.07, symmetric=True)

epochs = 30
checkpoint_pcts = [0.25, 0.50, 0.75]
checkpoint_epochs = [int(epochs * pct) for pct in checkpoint_pcts]

print(f"Training Transformer. Will save checkpoints at epochs: {checkpoint_epochs}")

for epoch in range(epochs):
    model_trans.train()
    epoch_loss = 0.0
    for batch in train_loader:
        z_img = batch['z_img'].to(device)
        z_aud = batch['z_aud'].to(device)
        v_teacher = batch['v_teacher'].to(device)
        
        optimizer.zero_grad()
        v_pred = model_trans(z_img, z_aud)
        
        # Joint loss
        loss_cos = cosine_loss_fn(v_pred, v_teacher)
        loss_nce = infonce_loss_fn(v_pred, v_teacher)
        loss = loss_cos + 0.5 * loss_nce
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * z_img.size(0)
        
    train_loss = epoch_loss / len(train_dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Joint Loss: {train_loss:.4f}")
        
    # Save intermediate checkpoints to models/ directory
    if (epoch + 1) in checkpoint_epochs:
        pct = int(((epoch + 1) / epochs) * 100)
        chk_path = f"models/transformer_fusion_{pct}pct.pt"
        os.makedirs("models", exist_ok=True)
        torch.save(model_trans.state_dict(), chk_path)
        print(f"Intermediate checkpoint saved: {chk_path} at epoch {epoch+1}")

# Save the final model weights to models/ directory
os.makedirs("models", exist_ok=True)
torch.save(model_trans.state_dict(), "models/transformer_fusion.pt")
print("Transformer training complete and weights saved to models/transformer_fusion.pt!")


Training Transformer. Will save checkpoints at epochs: [7, 15, 22]


Epoch 01/30 | Train Joint Loss: 1.8723


Epoch 05/30 | Train Joint Loss: 0.7798


Intermediate checkpoint saved: models/transformer_fusion_23pct.pt at epoch 7


Epoch 10/30 | Train Joint Loss: 0.6046


Epoch 15/30 | Train Joint Loss: 0.5210
Intermediate checkpoint saved: models/transformer_fusion_50pct.pt at epoch 15


Epoch 20/30 | Train Joint Loss: 0.4707


Intermediate checkpoint saved: models/transformer_fusion_73pct.pt at epoch 22


Epoch 25/30 | Train Joint Loss: 0.4432


Epoch 30/30 | Train Joint Loss: 0.4215
Transformer training complete and weights saved to models/transformer_fusion.pt!
